In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F
from pyspark.sql.window import Window

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 136, Finished, Available, Finished, True)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 137, Finished, Available, Finished, True)

+----------------------------------+------------------------+-----------+
|namespace                         |tableName               |isTemporary|
+----------------------------------+------------------------+-----------+
|RetailPipeline.RetailLakehouse.dbo|agg_daily_sales         |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_date                |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_product             |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_store               |false      |
|RetailPipeline.RetailLakehouse.dbo|dq_results              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales_unpartitioned|false      |
|RetailPipeline.RetailLakehouse.dbo|ingestion_control       |false      |
|RetailPipeline.RetailLakehouse.dbo|inventory               |false      |
|RetailPipeline.RetailLakehouse.dbo|pipeline_run_log        |false      |
|RetailPipeline.RetailLakehouse.dbo|ra

In [ ]:
print("RAW TABLE")
spark.table("raw").printSchema()

print("RAW ROW COUNT:")
print(spark.table("raw").count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 138, Finished, Available, Finished, True)

RAW TABLE
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)

RAW ROW COUNT:
505200


In [ ]:
print("INVENTORY TABLE")
spark.table("inventory").printSchema()

print("INVENTORY ROW COUNT:")
print(spark.table("inventory").count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 139, Finished, Available, Finished, False)

INVENTORY TABLE
root
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- StockQty: string (nullable = true)
 |-- SnapshotDate: string (nullable = true)
 |-- WarehouseZone: string (nullable = true)

INVENTORY ROW COUNT:
50000


In [ ]:
spark.table("raw").show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 140, Finished, Available, Finished, False)

+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North    |Thomasville    |2025-06-28|NULL          |NULL       |
|STR-0005|North Bonnie Store 5     |North    |West Connor    |2023-06-19|NULL          |NULL       |
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
only showing top 5 rows



In [ ]:
spark.table("inventory").show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 141, Finished, Available, Finished, False)

+--------+---------+------------+---------+--------+------------+-------------+
|StoreID |ProductID|ProductName |Category |StockQty|SnapshotDate|WarehouseZone|
+--------+---------+------------+---------+--------+------------+-------------+
|STR-0156|PRD-0363 |Will Bag    |Household|113     |2025-01-09  |A2           |
|STR-0151|PRD-0324 |Minute Bag  |Frozen   |106     |2025-01-09  |A1           |
|STR-0168|PRD-0350 |Once Bag    |Dairy    |277     |2025-01-09  |B1           |
|STR-0056|PRD-0122 |Official Bag|Snacks   |429     |2025-01-09  |B2           |
|STR-0002|PRD-0302 |Reason Bag  |Frozen   |325     |2025-01-09  |C1           |
+--------+---------+------------+---------+--------+------------+-------------+
only showing top 5 rows



In [ ]:
mssparkutils.fs.ls("Files")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 142, Finished, Available, Finished, False)

[FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00000-b10d2f93-c5e5-416a-ac56-a3a4e6d6ae1d-c000.snappy.txt, name=part-00000-b10d2f93-c5e5-416a-ac56-a3a4e6d6ae1d-c000.snappy.txt, size=105),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-6038e427-edc6-4e5c-968b-5fdd010c6ec6-c000.snappy.txt, name=part-00001-6038e427-edc6-4e5c-968b-5fdd010c6ec6-c000.snappy.txt, size=101),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-68c203bb-a229-4c86-99cd-e7e7a2f09cca-c000.snappy.txt, name=part-00001-68c203bb-a229-4c86-99cd-e7e7a2f09cca-c000.snappy.txt, size=161),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-f98a1e3

In [ ]:
mssparkutils.fs.ls("Files/raw")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 143, Finished, Available, Finished, False)

[FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/raw/inventory, name=inventory, size=0),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/raw/sales_events.csv, name=sales_events.csv, size=33919978),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/raw/store_master.csv, name=store_master.csv, size=12561)]

In [ ]:
print("===== TABLES =====")
spark.sql("SHOW TABLES").show(truncate=False)

print("===== RAW =====")
spark.table("raw").printSchema()
print("Rows:", spark.table("raw").count())
spark.table("raw").show(5, truncate=False)

print("===== INVENTORY =====")
spark.table("inventory").printSchema()
print("Rows:", spark.table("inventory").count())
spark.table("inventory").show(5, truncate=False)

print("===== FILES =====")
mssparkutils.fs.ls("Files")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 144, Finished, Available, Finished, False)

===== TABLES =====
+----------------------------------+------------------------+-----------+
|namespace                         |tableName               |isTemporary|
+----------------------------------+------------------------+-----------+
|RetailPipeline.RetailLakehouse.dbo|agg_daily_sales         |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_date                |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_product             |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_store               |false      |
|RetailPipeline.RetailLakehouse.dbo|dq_results              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales_unpartitioned|false      |
|RetailPipeline.RetailLakehouse.dbo|ingestion_control       |false      |
|RetailPipeline.RetailLakehouse.dbo|inventory               |false      |
|RetailPipeline.RetailLakehouse.dbo|pipeline_run_log        |false      |
|RetailPipeline.Ret

[FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00000-b10d2f93-c5e5-416a-ac56-a3a4e6d6ae1d-c000.snappy.txt, name=part-00000-b10d2f93-c5e5-416a-ac56-a3a4e6d6ae1d-c000.snappy.txt, size=105),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-6038e427-edc6-4e5c-968b-5fdd010c6ec6-c000.snappy.txt, name=part-00001-6038e427-edc6-4e5c-968b-5fdd010c6ec6-c000.snappy.txt, size=101),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-68c203bb-a229-4c86-99cd-e7e7a2f09cca-c000.snappy.txt, name=part-00001-68c203bb-a229-4c86-99cd-e7e7a2f09cca-c000.snappy.txt, size=161),
 FileInfo(path=abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.microsoft.com/51f61e23-fb0a-4159-bc80-f4f995e9d2d8/Files/part-00001-f98a1e3

In [ ]:
inventory = spark.table("inventory")

inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw_inventory")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 145, Finished, Available, Finished, False)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 146, Finished, Available, Finished, False)

+----------------------------------+------------------------+-----------+
|namespace                         |tableName               |isTemporary|
+----------------------------------+------------------------+-----------+
|RetailPipeline.RetailLakehouse.dbo|agg_daily_sales         |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_date                |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_product             |false      |
|RetailPipeline.RetailLakehouse.dbo|dim_store               |false      |
|RetailPipeline.RetailLakehouse.dbo|dq_results              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales              |false      |
|RetailPipeline.RetailLakehouse.dbo|fact_sales_unpartitioned|false      |
|RetailPipeline.RetailLakehouse.dbo|ingestion_control       |false      |
|RetailPipeline.RetailLakehouse.dbo|inventory               |false      |
|RetailPipeline.RetailLakehouse.dbo|pipeline_run_log        |false      |
|RetailPipeline.RetailLakehouse.dbo|ra

In [ ]:
raw_inventory = spark.table("raw_inventory")

print("Raw inventory rows:", raw_inventory.count())

raw_inventory.printSchema()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 147, Finished, Available, Finished, False)

Raw inventory rows: 50000
root
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- StockQty: string (nullable = true)
 |-- SnapshotDate: string (nullable = true)
 |-- WarehouseZone: string (nullable = true)



In [ ]:
inventory = spark.table("raw_inventory")

print("Inventory rows:", inventory.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 148, Finished, Available, Finished, False)

Inventory rows: 50000


In [ ]:
from pyspark.sql import functions as F

inventory.groupBy("SnapshotDate").agg(
    F.count("*").alias("rows"),
    F.count("WarehouseZone").alias("warehouse_zone_filled")
).orderBy("SnapshotDate").show(20, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 149, Finished, Available, Finished, False)

+------------+----+---------------------+
|SnapshotDate|rows|warehouse_zone_filled|
+------------+----+---------------------+
|2025-01-01  |5000|0                    |
|2025-01-02  |5000|0                    |
|2025-01-03  |5000|0                    |
|2025-01-04  |5000|0                    |
|2025-01-05  |5000|0                    |
|2025-01-06  |5000|5000                 |
|2025-01-07  |5000|5000                 |
|2025-01-08  |5000|5000                 |
|2025-01-09  |5000|5000                 |
|2025-01-10  |5000|5000                 |
+------------+----+---------------------+



In [ ]:
print("RAW ROW COUNT:")
print(spark.table("raw").count())

print("\nRAW SCHEMA:")
spark.table("raw").printSchema()

print("\nRAW SAMPLE:")
spark.table("raw").show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 150, Finished, Available, Finished, False)

RAW ROW COUNT:
505200

RAW SCHEMA:
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)


RAW SAMPLE:
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North    |Th

In [ ]:
tables = [
    "ingestion_control",
    "inventory",
    "raw",
    "raw_inventory"
]

for table in tables:
    print(
        table,
        "=>",
        spark.table(table).count()
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 151, Finished, Available, Finished, False)

ingestion_control => 12
inventory => 50000
raw => 505200
raw_inventory => 50000


In [ ]:
raw = spark.table("raw")

print("ROWS:", raw.count())

print("\n===== SCHEMA =====")
raw.printSchema()

print("\n===== SAMPLE =====")
raw.show(10, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 152, Finished, Available, Finished, False)

ROWS: 505200

===== SCHEMA =====
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)


===== SAMPLE =====
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North  

In [ ]:
raw.select("StoreID").groupBy("StoreID").count().orderBy("count").show(20)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 153, Finished, Available, Finished, False)

+--------------------+-----+
|             StoreID|count|
+--------------------+-----+
|  Bushport Store 102|    1|
| Snyderfurt Store 72|    1|
|West Conniefort S...|    1|
|Benjamintown Stor...|    1|
|New Karaberg Stor...|    1|
|Henryborough Stor...|    1|
|Houstonside Store 69|    1|
|  Singhbury Store 27|    1|
|Lisaborough Store 91|    1|
| Flowerston Store 37|    1|
|Amandachester Sto...|    1|
|South Paula Store 52|    1|
|Port Karenport St...|    1|
|  Snowland Store 187|    1|
| West Ryan Store 147|    1|
|Rosarioberg Store 16|    1|
|North Johnathan S...|    1|
|Carterhaven Store 75|    1|
|North Rhonda Stor...|    1|
| Aliciamouth Store 8|    1|
+--------------------+-----+
only showing top 20 rows



In [ ]:
raw = spark.table("raw")

print("===== SCHEMA =====")
raw.printSchema()

print("===== FIRST 5 ROWS =====")
raw.show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 154, Finished, Available, Finished, False)

===== SCHEMA =====
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)

===== FIRST 5 ROWS =====
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North    |Thomas

In [ ]:
from pyspark.sql import functions as F

raw.groupBy(
    F.when(
        F.col("EventID").isNull(),
        "STORE_MASTER"
    ).otherwise("SALES")
    .alias("record_type")
).count().show()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 155, Finished, Available, Finished, False)

+-----------+------+
|record_type| count|
+-----------+------+
|      SALES|505200|
+-----------+------+



In [ ]:
raw_sales_events = (
    raw
    .filter(F.col("EventID").isNotNull())
)

print("Sales rows:", raw_sales_events.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 156, Finished, Available, Finished, False)

Sales rows: 505200


In [ ]:
raw_sales_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw_sales_events")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 157, Finished, Available, Finished, False)

In [ ]:
raw_store_master = (
    raw
    .filter(F.col("EventID").isNull())
)

print("Store rows:", raw_store_master.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 158, Finished, Available, Finished, False)

Store rows: 0


In [ ]:
raw = spark.table("raw")

print("===== SCHEMA =====")
raw.printSchema()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 159, Finished, Available, Finished, False)

===== SCHEMA =====
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)



In [ ]:
raw.show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 160, Finished, Available, Finished, False)

+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North    |Thomasville    |2025-06-28|NULL          |NULL       |
|STR-0005|North Bonnie Store 5     |North    |West Connor    |2023-06-19|NULL          |NULL       |
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
only showing top 5 rows



In [ ]:
from pyspark.sql import functions as F

raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in raw.columns
]).show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 161, Finished, Available, Finished, False)

+-------+-------+---------+--------+---------+--------------+-----------+
|EventID|StoreID|ProductID|Quantity|UnitPrice|EventTimestamp|ChannelType|
+-------+-------+---------+--------+---------+--------------+-----------+
|0      |0      |3788     |0       |0        |200           |200        |
+-------+-------+---------+--------+---------+--------------+-----------+



In [ ]:
raw.select(
    "EventID",
    "StoreID",
    "ProductID",
    "EventTimestamp"
).show(20, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 162, Finished, Available, Finished, False)

+--------+-------------------------+---------+--------------+
|EventID |StoreID                  |ProductID|EventTimestamp|
+--------+-------------------------+---------+--------------+
|STR-0001|South Debra Store 1      |East     |NULL          |
|STR-0002|North Christopher Store 2|South    |NULL          |
|STR-0003|Port Arianabury Store 3  |West     |NULL          |
|STR-0004|New Micheleport Store 4  |North    |NULL          |
|STR-0005|North Bonnie Store 5     |North    |NULL          |
|STR-0006|New Jamieside Store 6    |Central  |NULL          |
|STR-0007|Wardbury Store 7         |North    |NULL          |
|STR-0008|Aliciamouth Store 8      |East     |NULL          |
|STR-0009|Martinmouth Store 9      |Central  |NULL          |
|STR-0010|Port Angelamouth Store 10|North    |NULL          |
|STR-0011|Jonesland Store 11       |Central  |NULL          |
|STR-0012|Deborahmouth Store 12    |South    |NULL          |
|STR-0013|Kellyville Store 13      |North    |NULL          |
|STR-001

In [ ]:
raw.printSchema()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 163, Finished, Available, Finished, False)

root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)



In [ ]:
raw.show(5, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 164, Finished, Available, Finished, False)

+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|EventID |StoreID                  |ProductID|Quantity       |UnitPrice |EventTimestamp|ChannelType|
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
|STR-0001|South Debra Store 1      |East     |Port Amandastad|2022-03-25|NULL          |NULL       |
|STR-0002|North Christopher Store 2|South    |Jonesland      |2023-07-11|NULL          |NULL       |
|STR-0003|Port Arianabury Store 3  |West     |New Brendaton  |2023-05-14|NULL          |NULL       |
|STR-0004|New Micheleport Store 4  |North    |Thomasville    |2025-06-28|NULL          |NULL       |
|STR-0005|North Bonnie Store 5     |North    |West Connor    |2023-06-19|NULL          |NULL       |
+--------+-------------------------+---------+---------------+----------+--------------+-----------+
only showing top 5 rows



In [ ]:
raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in raw.columns
]).show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 165, Finished, Available, Finished, False)

+-------+-------+---------+--------+---------+--------------+-----------+
|EventID|StoreID|ProductID|Quantity|UnitPrice|EventTimestamp|ChannelType|
+-------+-------+---------+--------+---------+--------------+-----------+
|0      |0      |3788     |0       |0        |200           |200        |
+-------+-------+---------+--------+---------+--------------+-----------+



In [ ]:
raw.select(
    "EventID",
    "StoreID",
    "ProductID",
    "EventTimestamp"
).show(20, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 166, Finished, Available, Finished, False)

+--------+-------------------------+---------+--------------+
|EventID |StoreID                  |ProductID|EventTimestamp|
+--------+-------------------------+---------+--------------+
|STR-0001|South Debra Store 1      |East     |NULL          |
|STR-0002|North Christopher Store 2|South    |NULL          |
|STR-0003|Port Arianabury Store 3  |West     |NULL          |
|STR-0004|New Micheleport Store 4  |North    |NULL          |
|STR-0005|North Bonnie Store 5     |North    |NULL          |
|STR-0006|New Jamieside Store 6    |Central  |NULL          |
|STR-0007|Wardbury Store 7         |North    |NULL          |
|STR-0008|Aliciamouth Store 8      |East     |NULL          |
|STR-0009|Martinmouth Store 9      |Central  |NULL          |
|STR-0010|Port Angelamouth Store 10|North    |NULL          |
|STR-0011|Jonesland Store 11       |Central  |NULL          |
|STR-0012|Deborahmouth Store 12    |South    |NULL          |
|STR-0013|Kellyville Store 13      |North    |NULL          |
|STR-001

In [ ]:
from pyspark.sql import functions as F

raw = spark.table("raw")

raw_store_master = raw.filter(
    F.col("EventTimestamp").isNull()
)

print("Store Master rows:", raw_store_master.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 167, Finished, Available, Finished, False)

Store Master rows: 200


In [ ]:
raw_sales_events = raw.filter(
    F.col("EventTimestamp").isNotNull()
)

print("Sales rows:", raw_sales_events.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 168, Finished, Available, Finished, False)

Sales rows: 505000


In [ ]:
print("Original raw:", raw.count())
print("Store Master:", raw_store_master.count())
print("Sales:", raw_sales_events.count())

print(
    "Store + Sales:",
    raw_store_master.count() + raw_sales_events.count()
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 169, Finished, Available, Finished, False)

Original raw: 505200
Store Master: 200
Sales: 505000
Store + Sales: 505200


In [ ]:
raw_store_master.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw_store_master")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 170, Finished, Available, Finished, False)

In [ ]:
raw_sales_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("raw_sales_events")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 171, Finished, Available, Finished, False)

In [ ]:
for table in [
    "raw_inventory",
    "raw_sales_events",
    "raw_store_master"
]:
    print(
        table,
        "=>",
        spark.table(table).count()
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 172, Finished, Available, Finished, False)

In [ ]:
spark.table("raw_store_master").show(
    10,
    truncate=False
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 173, Finished, Available, Finished, False)

In [ ]:
spark.table("raw_sales_events").show(
    5,
    truncate=False
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 174, Finished, Available, Finished, False)

In [ ]:
spark.table("raw_sales_events").select(
    "EventID",
    "StoreID",
    "ProductID",
    "EventTimestamp"
).show(10, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 175, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sales = spark.table("raw_sales_events")

print("Raw sales rows:", sales.count())

sales.printSchema()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 176, Finished, Available, Finished, False)

In [ ]:
duplicate_events = (
    sales
    .groupBy("EventID")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate EventIDs:",
    duplicate_events.count()
)

duplicate_events.show(10, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 177, Finished, Available, Finished, False)

In [ ]:
dedup_window = (
    Window
    .partitionBy("EventID")
    .orderBy(
        F.col("EventTimestamp").desc()
    )
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 178, Finished, Available, Finished, False)

In [ ]:
sales_deduped = (
    sales
    .withColumn(
        "rn",
        F.row_number().over(dedup_window)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

print("Before:", sales.count())
print("After:", sales_deduped.count())

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 179, Finished, Available, Finished, False)

In [ ]:
sales_clean = (
    sales_deduped
    .withColumn(
        "EventTimestamp",
        F.to_timestamp("EventTimestamp")
    )
    .withColumn(
        "SaleDate",
        F.to_date("EventTimestamp")
    )
)

sales_clean.select(
    "EventID",
    "EventTimestamp",
    "SaleDate"
).show(10, truncate=False)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 14, Finished, Available, Finished, False)

+-----------+-------------------+----------+
|EventID    |EventTimestamp     |SaleDate  |
+-----------+-------------------+----------+
|EVT-0000011|2025-01-01 10:27:40|2025-01-01|
|EVT-0000018|2025-01-01 09:26:32|2025-01-01|
|EVT-0000031|2025-01-01 09:20:29|2025-01-01|
|EVT-0000042|2025-01-01 04:23:03|2025-01-01|
|EVT-0000053|2025-01-01 04:42:31|2025-01-01|
|EVT-0000054|2025-01-01 04:39:18|2025-01-01|
|EVT-0000057|2025-01-01 23:53:16|2025-01-01|
|EVT-0000064|2025-01-01 22:46:04|2025-01-01|
|EVT-0000081|2025-01-03 00:00:00|2025-01-03|
|EVT-0000083|2025-01-01 22:23:11|2025-01-01|
+-----------+-------------------+----------+
only showing top 10 rows



In [ ]:
bad_dates = sales_clean.filter(
    F.col("SaleDate").isNull()
)

print(
    "Sales with invalid EventTimestamp:",
    bad_dates.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 15, Finished, Available, Finished, False)

Sales with invalid EventTimestamp: 0


In [ ]:
sales_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("SaleDate") \
    .saveAsTable("silver_sales")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 16, Finished, Available, Finished, False)

In [ ]:
silver_sales = spark.table("silver_sales")

print(
    "Silver sales rows:",
    silver_sales.count()
)

silver_sales.printSchema()

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 17, Finished, Available, Finished, False)

Silver sales rows: 500000
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)
 |-- SaleDate: date (nullable = true)



In [ ]:
store_master = spark.table("raw_store_master")

print(
    "Store Master rows:",
    store_master.count()
)

store_master.printSchema()

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 18, Finished, Available, Finished, False)

Store Master rows: 200
root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)



In [ ]:
from pyspark.sql.functions import broadcast

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 19, Finished, Available, Finished, False)

In [ ]:
sales_enriched = (
    silver_sales
    .join(
        broadcast(store_master),
        on="StoreID",
        how="left"
    )
)

print(
    "Enriched sales rows:",
    sales_enriched.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 20, Finished, Available, Finished, False)

Enriched sales rows: 500000


In [ ]:
silver_sales.join(
    broadcast(store_master),
    on="StoreID",
    how="left"
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 21, Finished, Available, Finished, False)

DataFrame[StoreID: string, EventID: string, ProductID: string, Quantity: string, UnitPrice: string, EventTimestamp: timestamp, ChannelType: string, SaleDate: date, EventID: string, ProductID: string, Quantity: string, UnitPrice: string, EventTimestamp: timestamp, ChannelType: string]

In [ ]:
unmatched_stores = (
    sales_enriched
    .filter(F.col("StoreID").isNull())
    .count()
)

print(
    "Unmatched stores:",
    unmatched_stores
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 22, Finished, Available, Finished, False)

Unmatched stores: 0


In [ ]:
from pyspark.sql import functions as F

store_master = spark.table("raw_store_master")

store_master.printSchema()

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 34, Finished, Available, Finished, False)

root
 |-- EventID: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- ChannelType: string (nullable = true)



In [ ]:
store_master.columns


StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 35, Finished, Available, Finished, False)

['EventID',
 'StoreID',
 'ProductID',
 'Quantity',
 'UnitPrice',
 'EventTimestamp',
 'ChannelType']

In [ ]:
dim_store = (
    store_master
    .select(
        F.col("EventID").alias("StoreID"),
        F.col("StoreID").alias("StoreName"),
        F.col("ProductID").alias("Region")
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 36, Finished, Available, Finished, False)

In [ ]:
dim_store.show(10, truncate=False)

print("Store rows:", dim_store.count())

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 37, Finished, Available, Finished, False)

+--------+-------------------------+-------+
|StoreID |StoreName                |Region |
+--------+-------------------------+-------+
|STR-0001|South Debra Store 1      |East   |
|STR-0002|North Christopher Store 2|South  |
|STR-0003|Port Arianabury Store 3  |West   |
|STR-0004|New Micheleport Store 4  |North  |
|STR-0005|North Bonnie Store 5     |North  |
|STR-0006|New Jamieside Store 6    |Central|
|STR-0007|Wardbury Store 7         |North  |
|STR-0008|Aliciamouth Store 8      |East   |
|STR-0009|Martinmouth Store 9      |Central|
|STR-0010|Port Angelamouth Store 10|North  |
+--------+-------------------------+-------+
only showing top 10 rows

Store rows: 200


In [ ]:
dim_store.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_store")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 38, Finished, Available, Finished, False)

In [ ]:
store_master = spark.table("dim_store")

store_master.printSchema()

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 39, Finished, Available, Finished, False)

root
 |-- StoreID: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- Region: string (nullable = true)



In [ ]:
sales_enriched = (
    silver_sales
    .join(
        broadcast(store_master),
        on="StoreID",
        how="left"
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 40, Finished, Available, Finished, False)

In [ ]:
sales_enriched = (
    silver_sales.alias("s")
    .join(
        broadcast(store_master).alias("st"),
        F.col("s.StoreID") == F.col("st.StoreID"),
        "left"
    )
    .select(
        F.col("s.*"),
        F.col("st.StoreName"),
        F.col("st.Region")
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 41, Finished, Available, Finished, False)

In [ ]:
print(
    "Enriched sales rows:",
    sales_enriched.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 42, Finished, Available, Finished, False)

Enriched sales rows: 500000


In [ ]:
unmatched_stores = (
    sales_enriched
    .filter(F.col("StoreName").isNull())
    .count()
)

print(
    "Unmatched stores:",
    unmatched_stores
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 43, Finished, Available, Finished, False)

Unmatched stores: 0


In [ ]:
fact_sales = (
    sales_enriched
    .withColumn(
        "SalesAmount",
        F.col("Quantity") * F.col("UnitPrice")
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 44, Finished, Available, Finished, False)

In [ ]:
fact_sales = fact_sales.select(
    "EventID",
    "StoreID",
    "StoreName",
    "Region",
    "ProductID",
    "EventTimestamp",
    "SaleDate",
    "Quantity",
    "UnitPrice",
    "SalesAmount"
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 45, Finished, Available, Finished, False)

In [ ]:
fact_sales.show(10, truncate=False)

print(
    "Fact rows:",
    fact_sales.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 46, Finished, Available, Finished, False)

+-----------+--------+---------------------------+-------+---------+-------------------+----------+--------+---------+------------------+
|EventID    |StoreID |StoreName                  |Region |ProductID|EventTimestamp     |SaleDate  |Quantity|UnitPrice|SalesAmount       |
+-----------+--------+---------------------------+-------+---------+-------------------+----------+--------+---------+------------------+
|EVT-0092320|STR-0119|Coxland Store 119          |East   |PRD-0075 |2025-01-07 03:58:18|2025-01-07|7       |18.15    |127.04999999999998|
|EVT-0092325|STR-0011|Jonesland Store 11         |Central|PRD-0338 |2025-01-07 14:28:27|2025-01-07|8       |120.55   |964.4             |
|EVT-0092326|STR-0119|Coxland Store 119          |East   |PRD-0283 |2025-01-07 02:47:42|2025-01-07|17      |51.6     |877.2             |
|EVT-0092328|STR-0159|New Denise Store 159       |South  |PRD-0331 |2025-01-07 00:27:29|2025-01-07|6       |136.01   |816.06            |
|EVT-0092330|STR-0155|West Victori

In [ ]:
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("SaleDate") \
    .saveAsTable("fact_sales")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 47, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F

inventory = spark.table("raw_inventory")

dim_product = (
    inventory
    .select(
        "ProductID",
        "ProductName",
        "Category"
    )
    .dropDuplicates(["ProductID"])
)

print("Products:", dim_product.count())

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 48, Finished, Available, Finished, False)

Products: 400


In [ ]:
dim_product.show(10, truncate=False)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 49, Finished, Available, Finished, False)

+---------+---------------+-------------+
|ProductID|ProductName    |Category     |
+---------+---------------+-------------+
|PRD-0001 |Meeting Pack   |Beverages    |
|PRD-0002 |Lose Bag       |Frozen       |
|PRD-0003 |Again Bag      |Frozen       |
|PRD-0004 |Pretty Bottle  |Beverages    |
|PRD-0005 |Town Box       |Beverages    |
|PRD-0006 |It Bottle      |Personal Care|
|PRD-0007 |National Bottle|Frozen       |
|PRD-0008 |Republican Box |Household    |
|PRD-0009 |Recent Pack    |Snacks       |
|PRD-0010 |Wonder Jar     |Dairy        |
+---------+---------------+-------------+
only showing top 10 rows



In [ ]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_product")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 50, Finished, Available, Finished, False)

In [ ]:
invalid_products = (
    fact_sales
    .select("ProductID")
    .distinct()
    .join(
        dim_product.select("ProductID").distinct(),
        on="ProductID",
        how="left_anti"
    )
)

print(
    "Products in Sales but missing from Inventory:",
    invalid_products.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 51, Finished, Available, Finished, False)

Products in Sales but missing from Inventory: 974


In [ ]:
invalid_stores = (
    fact_sales
    .select("StoreID")
    .distinct()
    .join(
        dim_store.select("StoreID").distinct(),
        on="StoreID",
        how="left_anti"
    )
)

print(
    "Stores in Sales but missing from Store Master:",
    invalid_stores.count()
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 52, Finished, Available, Finished, False)

Stores in Sales but missing from Store Master: 0


In [ ]:
dim_date = (
    fact_sales
    .select("SaleDate")
    .distinct()
    .withColumn(
        "Year",
        F.year("SaleDate")
    )
    .withColumn(
        "Month",
        F.month("SaleDate")
    )
    .withColumn(
        "MonthName",
        F.date_format("SaleDate", "MMMM")
    )
    .withColumn(
        "Quarter",
        F.quarter("SaleDate")
    )
    .withColumn(
        "DayOfWeek",
        F.dayofweek("SaleDate")
    )
    .withColumn(
        "DayName",
        F.date_format("SaleDate", "EEEE")
    )
)

print("Date rows:", dim_date.count())

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 53, Finished, Available, Finished, False)

Date rows: 30


In [ ]:
dim_date.orderBy("SaleDate").show(20, truncate=False)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 54, Finished, Available, Finished, False)

+----------+----+-----+---------+-------+---------+---------+
|SaleDate  |Year|Month|MonthName|Quarter|DayOfWeek|DayName  |
+----------+----+-----+---------+-------+---------+---------+
|2025-01-01|2025|1    |January  |1      |4        |Wednesday|
|2025-01-02|2025|1    |January  |1      |5        |Thursday |
|2025-01-03|2025|1    |January  |1      |6        |Friday   |
|2025-01-04|2025|1    |January  |1      |7        |Saturday |
|2025-01-05|2025|1    |January  |1      |1        |Sunday   |
|2025-01-06|2025|1    |January  |1      |2        |Monday   |
|2025-01-07|2025|1    |January  |1      |3        |Tuesday  |
|2025-01-08|2025|1    |January  |1      |4        |Wednesday|
|2025-01-09|2025|1    |January  |1      |5        |Thursday |
|2025-01-10|2025|1    |January  |1      |6        |Friday   |
|2025-01-11|2025|1    |January  |1      |7        |Saturday |
|2025-01-12|2025|1    |January  |1      |1        |Sunday   |
|2025-01-13|2025|1    |January  |1      |2        |Monday   |
|2025-01

In [ ]:
dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 55, Finished, Available, Finished, False)

In [ ]:
daily_sales = (
    fact_sales
    .groupBy(
        "SaleDate",
        "StoreID",
        "ProductID"
    )
    .agg(
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("SalesAmount").alias("TotalSales"),
        F.countDistinct("EventID").alias("TransactionCount")
    )
)

print("Daily sales rows:", daily_sales.count())

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 56, Finished, Available, Finished, False)

Daily sales rows: 404603


In [ ]:
daily_sales.orderBy(
    "SaleDate",
    "StoreID",
    "ProductID"
).show(20, truncate=False)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 57, Finished, Available, Finished, False)

+----------+--------+---------+-------------+------------------+----------------+
|SaleDate  |StoreID |ProductID|TotalQuantity|TotalSales        |TransactionCount|
+----------+--------+---------+-------------+------------------+----------------+
|2025-01-01|STR-0001|PRD-0006 |14.0         |329.84            |1               |
|2025-01-01|STR-0001|PRD-0016 |6.0          |646.92            |1               |
|2025-01-01|STR-0001|PRD-0020 |13.0         |1700.7900000000002|1               |
|2025-01-01|STR-0001|PRD-0039 |19.0         |1817.16           |1               |
|2025-01-01|STR-0001|PRD-0150 |6.0          |886.74            |1               |
|2025-01-01|STR-0001|PRD-0191 |14.0         |343.56            |1               |
|2025-01-01|STR-0001|PRD-0203 |16.0         |1090.88           |1               |
|2025-01-01|STR-0001|PRD-0217 |18.0         |423.54            |1               |
|2025-01-01|STR-0001|PRD-0223 |9.0          |528.48            |1               |
|2025-01-01|STR-

In [ ]:
rolling_window = (
    Window
    .partitionBy(
        "StoreID",
        "ProductID"
    )
    .orderBy(
        F.col("SaleDate").cast("long")
    )
    .rangeBetween(
        -6 * 86400,
        0
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 58, Finished, Available, Finished, False)

In [ ]:
daily_sales_rolling = (
    daily_sales
    .withColumn(
        "Rolling7DayAverage",
        F.avg("TotalSales").over(
            rolling_window
        )
    )
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 59, Finished, Available, Finished, False)

In [ ]:
daily_sales_rolling.orderBy(
    "StoreID",
    "ProductID",
    "SaleDate"
).show(20, truncate=False)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 60, Finished, Available, Finished, False)

+----------+--------+---------+-------------+------------------+----------------+------------------+
|SaleDate  |StoreID |ProductID|TotalQuantity|TotalSales        |TransactionCount|Rolling7DayAverage|
+----------+--------+---------+-------------+------------------+----------------+------------------+
|2025-01-05|STR-0001|NULL     |17.0         |2337.5            |1               |1495.47375        |
|2025-01-11|STR-0001|NULL     |26.0         |2254.68           |2               |1495.47375        |
|2025-01-15|STR-0001|NULL     |44.0         |2479.9            |3               |1495.47375        |
|2025-01-16|STR-0001|NULL     |17.0         |1661.3099999999997|2               |1495.47375        |
|2025-01-18|STR-0001|NULL     |3.0          |331.02            |1               |1495.47375        |
|2025-01-19|STR-0001|NULL     |15.0         |207.35999999999999|2               |1495.47375        |
|2025-01-22|STR-0001|NULL     |18.0         |1349.82           |1               |1495.47375

In [ ]:
daily_sales_rolling.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("SaleDate") \
    .saveAsTable("agg_daily_sales")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 61, Finished, Available, Finished, False)

In [100]:
gold_tables = [
    "fact_sales",
    "dim_store",
    "dim_product",
    "dim_date",
    "agg_daily_sales"
]

for table in gold_tables:
    print(
        table,
        "=>",
        spark.table(table).count()
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 102, Finished, Available, Finished, False)

fact_sales => 500000
dim_store => 200
dim_product => 400
dim_date => 30
agg_daily_sales => 404603


In [101]:
from pyspark.sql import functions as F

def run_data_quality_checks(
    df,
    table_name,
    key_columns=None,
    reference_checks=None
):
    
    results = []
    
    # -------------------------
    # 1. Row count
    # -------------------------
    row_count = df.count()
    
    results.append({
        "table": table_name,
        "check": "row_count",
        "value": row_count,
        "status": "PASS" if row_count > 0 else "FAIL"
    })
    
    # -------------------------
    # 2. Null checks
    # -------------------------
    if key_columns:
        
        for column in key_columns:
            
            null_count = (
                df.filter(
                    F.col(column).isNull()
                ).count()
            )
            
            results.append({
                "table": table_name,
                "check": f"null_{column}",
                "value": null_count,
                "status": "PASS" if null_count == 0 else "FAIL"
            })
    
    # -------------------------
    # 3. Referential integrity
    # -------------------------
    if reference_checks:
        
        for check in reference_checks:
            
            source_df = check["source_df"]
            source_column = check["source_column"]
            reference_df = check["reference_df"]
            reference_column = check["reference_column"]
            check_name = check["name"]
            
            invalid_count = (
                source_df
                .select(source_column)
                .distinct()
                .join(
                    reference_df
                    .select(reference_column)
                    .distinct(),
                    source_df[source_column] == reference_df[reference_column],
                    "left_anti"
                )
                .count()
            )
            
            results.append({
                "table": table_name,
                "check": check_name,
                "value": invalid_count,
                "status": "PASS" if invalid_count == 0 else "FAIL"
            })
    
    return results

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 103, Finished, Available, Finished, False)

In [102]:
dq_sales = run_data_quality_checks(
    df=fact_sales,
    table_name="fact_sales",
    key_columns=[
        "EventID",
        "StoreID",
        "ProductID",
        "SaleDate"
    ],
    reference_checks=[
        {
            "name": "sales_to_store",
            "source_df": fact_sales,
            "source_column": "StoreID",
            "reference_df": dim_store,
            "reference_column": "StoreID"
        },
        {
            "name": "sales_to_product",
            "source_df": fact_sales,
            "source_column": "ProductID",
            "reference_df": dim_product,
            "reference_column": "ProductID"
        }
    ]
)

dq_sales

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 104, Finished, Available, Finished, False)

[{'table': 'fact_sales',
  'check': 'row_count',
  'value': 500000,
  'status': 'PASS'},
 {'table': 'fact_sales',
  'check': 'null_EventID',
  'value': 0,
  'status': 'PASS'},
 {'table': 'fact_sales',
  'check': 'null_StoreID',
  'value': 0,
  'status': 'PASS'},
 {'table': 'fact_sales',
  'check': 'null_ProductID',
  'value': 3742,
  'status': 'FAIL'},
 {'table': 'fact_sales',
  'check': 'null_SaleDate',
  'value': 0,
  'status': 'PASS'},
 {'table': 'fact_sales',
  'check': 'sales_to_store',
  'value': 0,
  'status': 'PASS'},
 {'table': 'fact_sales',
  'check': 'sales_to_product',
  'value': 974,
  'status': 'FAIL'}]

In [103]:
for result in dq_sales:
    print(
        f"{result['check']:25} "
        f"value={result['value']} "
        f"status={result['status']}"
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 105, Finished, Available, Finished, False)

row_count                 value=500000 status=PASS
null_EventID              value=0 status=PASS
null_StoreID              value=0 status=PASS
null_ProductID            value=3742 status=FAIL
null_SaleDate             value=0 status=PASS
sales_to_store            value=0 status=PASS
sales_to_product          value=974 status=FAIL


In [104]:
total_sales = fact_sales.count()

invalid_products = (
    fact_sales
    .select("ProductID")
    .distinct()
    .join(
        dim_product.select("ProductID").distinct(),
        on="ProductID",
        how="left_anti"
    )
    .count()
)

invalid_product_rate = (
    invalid_products / total_sales * 100
)

print("Total sales:", total_sales)
print("Invalid product references:", invalid_products)
print(
    f"Invalid product rate: {invalid_product_rate:.4f}%"
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 106, Finished, Available, Finished, False)

Total sales: 500000
Invalid product references: 974
Invalid product rate: 0.1948%


In [105]:
DQ_THRESHOLD = 1.0

if invalid_product_rate > DQ_THRESHOLD:
    dq_status = "FAIL"
else:
    dq_status = "PASS"

print("DQ Threshold:", DQ_THRESHOLD, "%")
print("DQ Status:", dq_status)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 107, Finished, Available, Finished, False)

DQ Threshold: 1.0 %
DQ Status: PASS


In [106]:
dq_inventory = run_data_quality_checks(
    df=inventory,
    table_name="raw_inventory",
    key_columns=[
        "StoreID",
        "ProductID",
        "SnapshotDate"
    ]
)

for result in dq_inventory:
    print(
        f"{result['check']:25} "
        f"value={result['value']} "
        f"status={result['status']}"
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 108, Finished, Available, Finished, False)

row_count                 value=50000 status=PASS
null_StoreID              value=0 status=PASS
null_ProductID            value=0 status=PASS
null_SnapshotDate         value=0 status=PASS


In [107]:
dq_store = run_data_quality_checks(
    df=dim_store,
    table_name="dim_store",
    key_columns=[
        "StoreID",
        "StoreName",
        "Region"
    ]
)

for result in dq_store:
    print(
        f"{result['check']:25} "
        f"value={result['value']} "
        f"status={result['status']}"
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 109, Finished, Available, Finished, False)

row_count                 value=200 status=PASS
null_StoreID              value=0 status=PASS
null_StoreName            value=0 status=PASS
null_Region               value=0 status=PASS


In [108]:
all_dq_results = dq_sales + dq_inventory + dq_store

dq_df = spark.createDataFrame(all_dq_results)

dq_df.show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 110, Finished, Available, Finished, False)

+-----------------+------+-------------+------+
|check            |status|table        |value |
+-----------------+------+-------------+------+
|row_count        |PASS  |fact_sales   |500000|
|null_EventID     |PASS  |fact_sales   |0     |
|null_StoreID     |PASS  |fact_sales   |0     |
|null_ProductID   |FAIL  |fact_sales   |3742  |
|null_SaleDate    |PASS  |fact_sales   |0     |
|sales_to_store   |PASS  |fact_sales   |0     |
|sales_to_product |FAIL  |fact_sales   |974   |
|row_count        |PASS  |raw_inventory|50000 |
|null_StoreID     |PASS  |raw_inventory|0     |
|null_ProductID   |PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|row_count        |PASS  |dim_store    |200   |
|null_StoreID     |PASS  |dim_store    |0     |
|null_StoreName   |PASS  |dim_store    |0     |
|null_Region      |PASS  |dim_store    |0     |
+-----------------+------+-------------+------+



In [110]:
dq_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dq_results")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 112, Finished, Available, Finished, False)

In [111]:
spark.table("dq_results").show(
    50,
    truncate=False
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 113, Finished, Available, Finished, False)

+-----------------+------+-------------+------+
|check            |status|table        |value |
+-----------------+------+-------------+------+
|null_ProductID   |PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|null_ProductID   |PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|null_SnapshotDate|PASS  |raw_inventory|0     |
|sales_to_store   |PASS  |fact_sales   |0     |
|sales_to_product |FAIL  |fact_sales   |974   |
|sales_to_store   |PASS  |fact_sales   |0     |
|sales_to_product |FAIL  |fact_sales   |974   |
|null_ProductID   |FAIL  |fact_sales   |3742  |
|null_SaleDate    |PASS  |fact_sales   |0     |
|null_ProductID   |PASS  |raw_inventory|0     |
|null_ProductID   |PASS  |raw_inventory|0     |
|null_ProductID   |FAIL  |fact_sales   |3742  |
|null_SaleDate    |PASS  |fact_sales   |0     |
|sales_to_product |FAIL  |fact_sales   |

In [112]:
from pyspark.sql import types as T

log_schema = T.StructType([
    T.StructField("RunID", T.StringType(), True),
    T.StructField("Stage", T.StringType(), True),
    T.StructField("InputRows", T.LongType(), True),
    T.StructField("OutputRows", T.LongType(), True),
    T.StructField("RejectedRows", T.LongType(), True),
    T.StructField("DurationSeconds", T.DoubleType(), True),
    T.StructField("Status", T.StringType(), True),
    T.StructField("DQStatus", T.StringType(), True),
    T.StructField("RunTimestamp", T.TimestampType(), True)
])

empty_log = spark.createDataFrame([], log_schema)

empty_log.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_run_log")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 114, Finished, Available, Finished, False)

In [113]:
spark.table("pipeline_run_log").printSchema()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 115, Finished, Available, Finished, False)

root
 |-- RunID: string (nullable = true)
 |-- Stage: string (nullable = true)
 |-- InputRows: long (nullable = true)
 |-- OutputRows: long (nullable = true)
 |-- RejectedRows: long (nullable = true)
 |-- DurationSeconds: double (nullable = true)
 |-- Status: string (nullable = true)
 |-- DQStatus: string (nullable = true)
 |-- RunTimestamp: timestamp (nullable = true)



In [114]:
from datetime import datetime
import uuid

test_log = [{
    "RunID": str(uuid.uuid4()),
    "Stage": "Silver_Sales",
    "InputRows": 505000,
    "OutputRows": 500000,
    "RejectedRows": 5000,
    "DurationSeconds": 120.5,
    "Status": "SUCCESS",
    "DQStatus": "PASS",
    "RunTimestamp": datetime.now()
}]

test_log_df = spark.createDataFrame(
    test_log,
    schema=log_schema
)

test_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_run_log")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 116, Finished, Available, Finished, False)

In [115]:
spark.table("pipeline_run_log").show(
    truncate=False
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 117, Finished, Available, Finished, False)

+------------------------------------+------------+---------+----------+------------+---------------+-------+--------+--------------------------+
|RunID                               |Stage       |InputRows|OutputRows|RejectedRows|DurationSeconds|Status |DQStatus|RunTimestamp              |
+------------------------------------+------------+---------+----------+------------+---------------+-------+--------+--------------------------+
|edd9756d-1025-4dae-bde6-1c39604ac74e|Silver_Sales|505000   |500000    |5000        |120.5          |SUCCESS|PASS    |2026-08-13 04:53:17.962895|
+------------------------------------+------------+---------+----------+------------+---------------+-------+--------+--------------------------+



In [116]:
def write_pipeline_log(
    run_id,
    stage,
    input_rows,
    output_rows,
    rejected_rows,
    duration_seconds,
    status,
    dq_status
):
    
    log_record = [{
        "RunID": run_id,
        "Stage": stage,
        "InputRows": int(input_rows),
        "OutputRows": int(output_rows),
        "RejectedRows": int(rejected_rows),
        "DurationSeconds": float(duration_seconds),
        "Status": status,
        "DQStatus": dq_status,
        "RunTimestamp": datetime.now()
    }]
    
    log_df = spark.createDataFrame(
        log_record,
        schema=log_schema
    )
    
    log_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("pipeline_run_log")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 118, Finished, Available, Finished, False)

In [117]:
import time

run_id = str(uuid.uuid4())

start_time = time.time()

input_rows = sales.count()
output_rows = sales_deduped.count()

rejected_rows = input_rows - output_rows

duration = time.time() - start_time

write_pipeline_log(
    run_id=run_id,
    stage="Silver_Sales",
    input_rows=input_rows,
    output_rows=output_rows,
    rejected_rows=rejected_rows,
    duration_seconds=duration,
    status="SUCCESS",
    dq_status="PASS"
)

print("Log written successfully")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 119, Finished, Available, Finished, True)

Log written successfully


In [118]:
spark.table("pipeline_run_log") \
    .orderBy("RunTimestamp", ascending=False) \
    .show(10, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 120, Finished, Available, Finished, True)

+------------------------------------+------------+---------+----------+------------+-----------------+-------+--------+--------------------------+
|RunID                               |Stage       |InputRows|OutputRows|RejectedRows|DurationSeconds  |Status |DQStatus|RunTimestamp              |
+------------------------------------+------------+---------+----------+------------+-----------------+-------+--------+--------------------------+
|74249ea3-714e-46c2-9c0f-6ac26cd2e651|Silver_Sales|505000   |500000    |5000        |1.757603406906128|SUCCESS|PASS    |2026-08-13 04:53:24.360271|
|edd9756d-1025-4dae-bde6-1c39604ac74e|Silver_Sales|505000   |500000    |5000        |120.5            |SUCCESS|PASS    |2026-08-13 04:53:17.962895|
+------------------------------------+------------+---------+----------+------------+-----------------+-------+--------+--------------------------+



In [119]:
DQ_THRESHOLD = 1.0

invalid_product_rate = (
    invalid_products / total_sales * 100
)

if invalid_product_rate > DQ_THRESHOLD:
    pipeline_dq_status = "FAIL"
else:
    pipeline_dq_status = "PASS"

print("Invalid product rate:", invalid_product_rate)
print("DQ status:", pipeline_dq_status)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 121, Finished, Available, Finished, True)

Invalid product rate: 0.1948
DQ status: PASS


In [120]:
if pipeline_dq_status == "FAIL":
    raise Exception(
        f"Data Quality threshold exceeded. "
        f"Invalid product rate = "
        f"{invalid_product_rate:.4f}% "
        f"(threshold = {DQ_THRESHOLD}%). "
        f"Pipeline halted."
    )

print("DQ threshold passed. Pipeline can continue.")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 122, Finished, Available, Finished, True)

DQ threshold passed. Pipeline can continue.


In [121]:
from pyspark.sql import functions as F

invalid_product_rate = (
    invalid_products / total_sales * 100
)

DQ_THRESHOLD = 1.0

if invalid_product_rate > DQ_THRESHOLD:
    pipeline_dq_status = "FAIL"
else:
    pipeline_dq_status = "PASS"

print("Invalid product rate:", invalid_product_rate)
print("DQ status:", pipeline_dq_status)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 123, Finished, Available, Finished, True)

Invalid product rate: 0.1948
DQ status: PASS


In [122]:
control = spark.table("ingestion_control")

print("Rows:", control.count())
print("Columns:", control.columns)

control.show(20, truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 124, Finished, Available, Finished, True)

Rows: 12
Columns: ['source_file', 'source_type', 'target_table', 'enabled']
+-------------------+-----------+----------------+-------+
|source_file        |source_type|target_table    |enabled|
+-------------------+-----------+----------------+-------+
|inventory_day08.csv|inventory  |raw_inventory   |true   |
|inventory_day09.csv|inventory  |raw_inventory   |true   |
|inventory_day02.csv|inventory  |raw_inventory   |true   |
|inventory_day03.csv|inventory  |raw_inventory   |true   |
|inventory_day05.csv|inventory  |raw_inventory   |true   |
|inventory_day06.csv|inventory  |raw_inventory   |true   |
|store_master.csv   |store      |raw_store_master|true   |
|sales_events.csv   |sales      |raw_sales_events|true   |
|inventory_day07.csv|inventory  |raw_inventory   |true   |
|inventory_day01.csv|inventory  |raw_inventory   |true   |
|inventory_day10.csv|inventory  |raw_inventory   |true   |
|inventory_day04.csv|inventory  |raw_inventory   |true   |
+-------------------+-----------+------

In [123]:
from pyspark.sql import Row

control_data = [
    Row(
        source_file="inventory_day01.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day02.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day03.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day04.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day05.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day06.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day07.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day08.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day09.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="inventory_day10.csv",
        source_type="inventory",
        target_table="raw_inventory",
        enabled=True
    ),
    Row(
        source_file="store_master.csv",
        source_type="store",
        target_table="raw_store_master",
        enabled=True
    ),
    Row(
        source_file="sales_events.csv",
        source_type="sales",
        target_table="raw_sales_events",
        enabled=True
    )
]

control_df = spark.createDataFrame(control_data)

control_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ingestion_control")

print("Control table recreated successfully.")

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 125, Submitted, Running, Running, True)

In [ ]:
control = spark.table("ingestion_control")

print("Total control records:", control.count())

control.orderBy("source_type", "source_file") \
    .show(20, truncate=False)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
inventory_control = (
    control
    .filter(F.col("source_type") == "inventory")
)

print(
    "Inventory files:",
    inventory_control.count()
)

inventory_control.show(
    20,
    truncate=False
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
print(
    "Enabled files:",
    control.filter(
        F.col("enabled") == True
    ).count()
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark.table("raw_inventory").count()

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
unpartitioned = spark.table("fact_sales")

unpartitioned.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_sales_unpartitioned")

print(
    "Rows:",
    spark.table("fact_sales_unpartitioned").count()
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 130, Finished, Available, Finished, True)

Rows: 500000


In [ ]:
import time

start = time.time()

before_count = (
    spark.table("fact_sales_unpartitioned")
    .filter(
        F.col("SaleDate") == "2025-01-15"
    )
    .count()
)

before_time = time.time() - start

print("Rows returned:", before_count)
print(f"Before partitioning: {before_time:.4f} seconds")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 65, Finished, Available, Finished, False)

Rows returned: 56538
Before partitioning: 4.7418 seconds


In [ ]:
start = time.time()

after_count = (
    spark.table("fact_sales")
    .filter(
        F.col("SaleDate") == "2025-01-15"
    )
    .count()
)

after_time = time.time() - start

print("Rows returned:", after_count)
print(f"After partitioning: {after_time:.4f} seconds")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 66, Finished, Available, Finished, False)

Rows returned: 56538
After partitioning: 2.0436 seconds


In [ ]:
improvement = (
    (before_time - after_time)
    / before_time
    * 100
)

print(f"Before: {before_time:.4f} seconds")
print(f"After:  {after_time:.4f} seconds")
print(f"Improvement: {improvement:.2f}%")

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 67, Finished, Available, Finished, False)

Before: 4.7418 seconds
After:  2.0436 seconds
Improvement: 56.90%


In [ ]:
spark.sql("""
DESCRIBE DETAIL fact_sales
""").select(
    "format",
    "partitionColumns",
    "numFiles"
).show(truncate=False)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 134, Finished, Available, Finished, True)

+------+----------------+--------+
|format|partitionColumns|numFiles|
+------+----------------+--------+
|delta |[SaleDate]      |30      |
+------+----------------+--------+



In [ ]:
start = time.time()

normal_join = (
    silver_sales.alias("s")
    .join(
        store_master.alias("st"),
        F.col("s.StoreID") == F.col("st.StoreID"),
        "left"
    )
    .count()
)

normal_join_time = time.time() - start

print(
    f"Normal join: {normal_join_time:.4f} seconds"
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 135, Finished, Available, Finished, True)

Normal join: 0.4756 seconds


In [ ]:
start = time.time()

broadcast_join = (
    silver_sales.alias("s")
    .join(
        F.broadcast(store_master).alias("st"),
        F.col("s.StoreID") == F.col("st.StoreID"),
        "left"
    )
    .count()
)

broadcast_join_time = time.time() - start

print(
    f"Broadcast join: {broadcast_join_time:.4f} seconds"
)

StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 68, Finished, Available, Finished, False)

Broadcast join: 0.7052 seconds


In [ ]:
broadcast_test = (
    silver_sales.alias("s")
    .join(
        F.broadcast(store_master).alias("st"),
        F.col("s.StoreID") == F.col("st.StoreID"),
        "left"
    )
)

broadcast_test.explain()

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 183, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [StoreID#21954], [StoreID#23808], LeftOuter, BuildRight, false
   :- FileScan parquet spark_catalog.chimcobldhq2akj5ehgmir2gd5o6ar39dpiiakj5ehgmir2cc5lmaq3felpma9b4c9ng.silver_sales[EventID#21953,StoreID#21954,ProductID#21955,Quantity#21956,UnitPrice#21957,EventTimestamp#21958,ChannelType#21959,SaleDate#21960] Batched: true, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[abfss://1996a785-ba94-42fc-9d68-5c46197082f5@onelake.dfs.fabric.m..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<EventID:string,StoreID:string,ProductID:string,Quantity:string,UnitPrice:string,EventTimes...
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, true]),false), [plan_id=22043]
      +- FileScan parquet spark_catalog.chimcobldhq2akj5ehgmir2gd5o6ar39dpiiakj5ehgmir2cc5lmaq3felpma9b4c9ng.dim_store[StoreID#23808,StoreName#23809,Region#23810] Batched: true, DataFilters: 

In [ ]:
for table in [
    "fact_sales",
    "dim_store",
    "dim_product",
    "dim_date",
    "agg_daily_sales"
]:
    print(
        f"{table}: {spark.table(table).count()} rows"
    )

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 184, Finished, Available, Finished, False)

fact_sales: 500000 rows


dim_store: 200 rows
dim_product: 400 rows


dim_date: 30 rows
agg_daily_sales: 404603 rows


In [183]:
inventory_daily = (
    inventory
    .groupBy(
        "SnapshotDate",
        "StoreID",
        "ProductID"
    )
    .agg(
        F.sum("StockQty").alias("TotalInventory")
    )
)

inventory_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("SnapshotDate") \
    .saveAsTable("agg_daily_inventory")

print(
    "Inventory aggregate rows:",
    inventory_daily.count()
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 185, Finished, Available, Finished, False)

Inventory aggregate rows: 48445


In [184]:
spark.table("agg_daily_inventory").show(
    20,
    truncate=False
)

StatementMeta(, 4094ac22-0cc3-45fb-b5ac-f1399734f718, 186, Finished, Available, Finished, False)

+------------+--------+---------+--------------+
|SnapshotDate|StoreID |ProductID|TotalInventory|
+------------+--------+---------+--------------+
|2025-01-08  |STR-0058|PRD-0248 |9.0           |
|2025-01-08  |STR-0150|PRD-0330 |107.0         |
|2025-01-08  |STR-0171|PRD-0307 |329.0         |
|2025-01-08  |STR-0011|PRD-0134 |209.0         |
|2025-01-08  |STR-0049|PRD-0011 |233.0         |
|2025-01-08  |STR-0146|PRD-0068 |233.0         |
|2025-01-08  |STR-0194|PRD-0098 |409.0         |
|2025-01-08  |STR-0002|PRD-0138 |163.0         |
|2025-01-08  |STR-0065|PRD-0267 |147.0         |
|2025-01-08  |STR-0009|PRD-0080 |211.0         |
|2025-01-08  |STR-0005|PRD-0288 |278.0         |
|2025-01-08  |STR-0033|PRD-0249 |327.0         |
|2025-01-08  |STR-0145|PRD-0385 |319.0         |
|2025-01-08  |STR-0173|PRD-0195 |256.0         |
|2025-01-08  |STR-0030|PRD-0173 |245.0         |
|2025-01-08  |STR-0156|PRD-0054 |177.0         |
|2025-01-08  |STR-0131|PRD-0094 |188.0         |
|2025-01-08  |STR-00

In [ ]:
start = time.time()
before_count = (
    spark.table("fact_sales_unpartitioned")
    .filter(F.col("SaleDate") == "2025-01-15")
    .count()
)
before_time = time.time() - start
start = time.time()
after_count = (
    spark.table("fact_sales")
    .filter(F.col("SaleDate") == "2025-01-15")
    .count()
)
after_time = time.time() - start


StatementMeta(, a2a32488-1c3c-440a-a6ed-1486ff69fc01, 62, Finished, Available, Finished, False)

NameError: name 'time' is not defined